# Final ensemble — C2 + cumulant features

Produces `ensemble_0..4.pt`, `best_model.pt`, calibrated thresholds and the full
evaluation, all saved to Drive.

## Only run this if you are adding cumulant features

You already have a **passing** ensemble (`results_keep_c2`): C2 / `stft_keep_rows`,
all three judged classes over 80%, JAMMING precision 0.997. Retraining with the
same flags would just reproduce it with different seeds.

The one pending upgrade is `cumulant_features`. Measured on a single model against
the same data:

| | QPSK blended | LFM_RADAR AP | FHSS AP | JAMMING AP |
|---|---|---|---|---|
| baseline | 0.707 | 0.8296 | 0.8195 | 0.9680 |
| + cumulants | **0.786** | **0.8343** | **0.8204** | **0.9684** |

QPSK +0.079 with all three judged classes moving up — the only lever tested that
cost nothing. It should also let the calibrator hold the same recall at a **higher**
threshold, which is where the precision improvement would come from (QPSK currently
fires on 8,151 windows to catch 3,202 real ones).

**Keep `results_keep_c2` safe.** It is a passing submission. This run writes
somewhere else and you only switch if the numbers come out better.

## 1. Code — the branch matters

`main` does **not** have the cumulant code. It is on `qpsk-cumulant-features`.

In [ ]:
BRANCH = 'qpsk-cumulant-features'

%cd /content
!rm -rf sedicAI_NEXA
!git clone -q -b $BRANCH https://github.com/eavan127/sedicAI_NEXA.git
%cd /content/sedicAI_NEXA
!git log --oneline -3

In [ ]:
!pip install -q pyyaml h5py

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available()
      else 'CPU ONLY - stop and switch to a GPU runtime')

## 2. Turn both flags on

`train_ensemble.py` and `src.train` have no architecture switches — they build the
model from `configs/default.yaml`. So the flags have to be set in the config here.
That is safe: this clone is thrown away when the runtime ends, and nothing on your
machine changes.

In [ ]:
!sed -i 's/^  stft_keep_rows: .*/  stft_keep_rows: true/'     configs/default.yaml
!sed -i 's/^  cumulant_features: .*/  cumulant_features: true/' configs/default.yaml
!grep -n 'stft_freq_summary:\|stft_keep_rows:\|cumulant_features:' configs/default.yaml

**Check it took.** Expect `stft_keep_rows=True`, `cumulant_features=True`,
`fc1.in = 195` and about **182,666** parameters. If `fc1.in` is 192 the cumulant flag
did not apply — stop, do not spend 13 GPU-hours on the wrong architecture.

In [ ]:
from src.config import CFG, CLASSES
from src.models.amc_cnn import AMC_CNN
m = AMC_CNN(num_classes=len(CLASSES), input_len=CFG['signal']['window_len'])
print('stft_freq_summary =', m.stft_branch.freq_summary)
print('stft_keep_rows    =', m.stft_branch.keep_rows)
print('cumulant_features =', m.cumulant_branch is not None)
print('fc1.in_features   =', m.fc1.in_features, '   (192 = no cumulants, 195 = cumulants on)')
print('parameters        =', f'{sum(p.numel() for p in m.parameters()):,}')
assert m.stft_branch.keep_rows and m.cumulant_branch is not None, 'flags did not apply'
assert m.fc1.in_features == 195
del m
print('\nOK')

## 3. Data

From `MyDrive/sedic/eavan-retrain/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DATA = '/content/drive/MyDrive/sedic/eavan-retrain'
!mkdir -p data/processed
!cp $DRIVE_DATA/X.npy          data/processed/
!cp $DRIVE_DATA/y.npy          data/processed/
!cp $DRIVE_DATA/snr_labels.npy data/processed/
!ls -la data/processed/

In [ ]:
import numpy as np
from src.config import CFG, CLASSES
X = np.load('data/processed/X.npy', mmap_mode='r')
y = np.load('data/processed/y.npy')
snr = np.load('data/processed/snr_labels.npy')
assert X.ndim == 3 and X.shape[1] == 2, X.shape
assert y.shape[1] == len(CLASSES), y.shape
assert sorted(set(snr.tolist())) == [float(b) for b in sorted(CFG['snr_bins_db'])]
print('X', X.shape, '| y', y.shape, '| SNR bins match config')

In [ ]:
!python -m pytest -q 2>&1 | tail -5

## 4. Train

Five seeds, then the single model. Long — leave the tab open and visible; Colab
disconnects idle sessions.

Config `epochs` is 30. The saved checkpoint is always the **best-validation** one,
so overrunning costs time, not quality. Your C2 runs converged around epoch 6–10.

In [ ]:
!python scripts/train_ensemble.py --models 5

**Save the checkpoints to Drive now**, before anything else. Colab deletes
everything when the runtime ends, and this is the expensive part.

In [ ]:
OUT = '/content/drive/MyDrive/sedic/results_rows_cum'
!mkdir -p $OUT
!cp results/ensemble_*.pt $OUT/
!ls -la $OUT/

In [ ]:
!python -m src.train

In [ ]:
!cp results/best_model.pt $OUT/ && ls -la $OUT/

## 5. Calibrate, then evaluate

Order matters. `evaluate.py` reads the thresholds from the config, so calibrating
first and pasting the values in means the scorecard reflects what you would actually
submit. Running evaluate before this gives you numbers at the *old* thresholds,
which were fitted to a different architecture.

In [ ]:
!python scripts/calibrate_thresholds.py --ensemble --n-models 5

Paste the printed values into `configs/default.yaml` under
`multilabel_thresholds_per_class`, then run the next cell. Edit the dictionary below
to match what calibration printed.

In [ ]:
# Paste the calibrated values here, then this writes them into the config.
NEW_THRESHOLDS = {
    'BPSK': 0.16, 'QPSK': 0.16, '16QAM': 0.19, '64QAM': 0.20,
    'LFM_RADAR': 0.24, 'FHSS': 0.24, 'JAMMING': 0.89, 'NOISE_FLOOR': 0.16,
}

import re, pathlib, json
p = pathlib.Path('configs/default.yaml')
text = p.read_text()
for cls, v in NEW_THRESHOLDS.items():
    text = re.sub(rf'^(\s+){cls}: [0-9.]+$', rf'\g<1>{cls}: {v}', text,
                  count=1, flags=re.M)
p.write_text(text)
pathlib.Path(f'{OUT}/thresholds_rows_cum.json').write_text(json.dumps(NEW_THRESHOLDS, indent=2))
!grep -n -A 10 'multilabel_thresholds_per_class:' configs/default.yaml | head -14

In [ ]:
!python -m src.evaluate --ensemble --n-models 5

In [ ]:
from IPython.display import Image, display
display(Image('evals/confusion_matrix.png'))
display(Image('evals/accuracy_vs_snr.png'))

## 6. Did it beat the one you already have?

`results_keep_c2` is the bar. It passed with LFM_RADAR 0.8415, FHSS 0.8291,
JAMMING 0.8399, and JAMMING precision 0.997.

**Switch only if all three judged classes still pass and nothing important went
backwards.** A QPSK gain is not worth a judged-class regression — QPSK is not
scored.

In [ ]:
import json, pathlib
d = json.loads(pathlib.Path('evals/ensemble_scorecard.json').read_text())
prev = {'LFM_RADAR': 0.8415, 'FHSS': 0.8291, 'JAMMING': 0.8399}

print(f"{'class':<12} {'new':>8} {'keep_c2':>9} {'delta':>8}   verdict")
print('-' * 52)
for c, old in prev.items():
    new = d['benchmark']['judged'][c]['recall']
    flag = 'PASS' if new >= 0.80 else 'FAIL'
    print(f'{c:<12} {new:>8.4f} {old:>9.4f} {new-old:>+8.4f}   {flag}')
print('\noverall benchmark:', 'PASSED' if d['benchmark']['passed'] else 'FAILED')
print()
print(f"{'class':<12} {'recall':>8} {'precision':>10}")
print('-' * 32)
for c, v in d['per_class'].items():
    print(f"{c:<12} {v['recall']:>8.4f} {v['precision']:>10.4f}")

In [ ]:
!cp -r evals $OUT/
!cp configs/default.yaml $OUT/default_rows_cum.yaml
!ls -la $OUT/

## 7. Back on your machine

If you are switching to this ensemble:

1. Copy `ensemble_*.pt` and `best_model.pt` from `sedic/results_rows_cum/` into
   `results/`
2. **In the same step**, set `stft_keep_rows: true` and `cumulant_features: true`
   in your local `configs/default.yaml`, and paste in the calibrated thresholds
   (`default_rows_cum.yaml` in that folder is the exact config this run used)
3. `python -m src.evaluate --ensemble --n-models 5` to confirm it reproduces

Config and checkpoints move **together**. With `cumulant_features: true`, `fc1` is
195 wide and no older checkpoint will load — `src/evaluate.py`,
`calibrate_thresholds.py`, the UI console and the static site build all construct a
model from that config.

If the numbers came out worse, change nothing. `results_keep_c2` stays the
submission.